In [1]:
import os

is_databricks = "dbutils" in globals()

bootstrap = os.getenv("KAFKA_BOOTSTRAP", "")
api_key = os.getenv("KAFKA_API_KEY", "")
api_secret = os.getenv("KAFKA_API_SECRET", "")
topic = os.getenv("KAFKA_TOPIC", "orders.events")

if is_databricks:
    print("Ambiente: Databricks")
    try:
        bootstrap = bootstrap or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_BOOTSTRAP")
        api_key = api_key or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_API_KEY")
        api_secret = api_secret or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_API_SECRET")
        topic = topic or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_TOPIC")
    except Exception as e:
        print("Aviso: não foi possível ler secrets do Databricks:", e)
else:
    from dotenv import load_dotenv
    load_dotenv()
    print("Ambiente: Local")
    print("Variáveis de ambiente carregadas de .env")

if not all([bootstrap, api_key, api_secret]):
    raise ValueError(
        "Configure as credenciais de Kafka em variáveis de ambiente ou em Databricks secrets."
    )

print("Configurações de Kafka carregadas com sucesso.")
print(f"Tópico: {topic}")

Ambiente: Local
Variáveis de ambiente carregadas de .env
Configurações de Kafka carregadas com sucesso.
Tópico: orders.events


In [7]:
if is_databricks:
    %pip install confluent-kafka
    dbutils.library.restartPython()
else:
    %pip install confluent-kafka

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Notebook Kafka Seguro
#
# Este notebook deve usar variáveis de ambiente ou Databricks secrets para credenciais.
#
# Não coloque `api_key` ou `api_secret` diretamente no arquivo.


# Ambiente unificado

A célula anterior já carrega as credenciais usando `.env` localmente ou `dbutils.secrets` no Databricks. A partir daqui, use `bootstrap`, `api_key`, `api_secret` e `topic` normalmente.

In [4]:
from confluent_kafka import Consumer
import json

config = {
    'bootstrap.servers': bootstrap,
    'group.id': 'connect-lcc',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': api_key,
    'sasl.password': api_secret,
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': True
}

consumer = Consumer(config)
consumer.subscribe([topic])
print(f"🔍 Consumindo '{topic}'... (5 mensagens ou Ctrl+Enter)")

count = 0
try:
    while count < 5:
        msg = consumer.poll(1.0)
        if msg is None:
            continue
        if msg.error():
            print(f"❌ Erro: {msg.error()}")
            break
        else:
            value = msg.value().decode('utf-8')
            print(f"✅ Mensagem {count+1}: {json.dumps(json.loads(value), indent=2)}")
            count += 1
except Exception as e:
    print(f"Erro: {e}")
finally:
    consumer.close()
    print("Consumer fechado")


🔍 Consumindo 'orders.events'... (5 mensagens ou Ctrl+Enter)
✅ Mensagem 1: {
  "order_id": "ord_00422",
  "customer_id": "cust_009",
  "amount": 440.66,
  "currency": "BRL",
  "status": "created",
  "items": [
    {
      "sku": "helmet_red",
      "qty": 2,
      "unit_price": 220.33
    }
  ],
  "source": "mobile",
  "event_time": "2026-04-10T14:19:54.766353+00:00"
}
✅ Mensagem 2: {
  "order_id": "ord_00423",
  "customer_id": "cust_018",
  "amount": 956.04,
  "currency": "BRL",
  "status": "shipped",
  "items": [
    {
      "sku": "jacket_team",
      "qty": 3,
      "unit_price": 318.68
    }
  ],
  "source": "mobile",
  "event_time": "2026-04-10T14:19:56.928403+00:00"
}
✅ Mensagem 3: {
  "order_id": "ord_00424",
  "customer_id": "cust_010",
  "amount": 434.86,
  "currency": "BRL",
  "status": "shipped",
  "items": [
    {
      "sku": "gloves_pro",
      "qty": 2,
      "unit_price": 217.43
    }
  ],
  "source": "mobile",
  "event_time": "2026-04-10T14:19:59.059199+00:00"
}
✅ Mens

In [9]:
from confluent_kafka import Consumer
import time

config = {
    'bootstrap.servers': bootstrap,
    'group.id': 'connect-lcc-latest',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': api_key,
    'sasl.password': api_secret,
    'auto.offset.reset': 'latest'
}
consumer = Consumer(config)
consumer.subscribe([topic])

print("⏳ Aguardando 10s por novas mensagens...")

msgs = []
start = time.time()
while time.time() - start < 10:
    msg = consumer.poll(1.0)
    if msg is None:
        continue
    if msg.error():
        print(f"❌ Erro: {msg.error()}")
        break
    else:
        msgs.append(msg.value().decode('utf-8'))
        print(f"🆕 NOVA: {msgs[-1][:100]}")

consumer.close()

if msgs:
    print(f"✅ {len(msgs)} novas mensagens recebidas!")
else:
    print("❌ Nenhuma nova mensagem em 10s")


KeyboardInterrupt: 